In [ ]:
import os, sys, json, gc, subprocess
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
TOKEN, HF_TOKEN = secrets.get_secret("GITHUB_TOKEN"), secrets.get_secret("HF_TOKEN")
PLAIN_URL = secrets.get_secret("GITHUB_REPO_URL")
REPO_URL = PLAIN_URL.replace("https://", f"https://{TOKEN}@")
REPO_DIR = "/kaggle/working/qwen2.5-sft-dpo-alignment"

def run(cmd):
    out = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    text = (out.stdout + out.stderr).replace(TOKEN, "***")
    print(text)
    if out.returncode != 0:
        raise RuntimeError(text)

def push(msg, notebook=None):
    """Optionally save this notebook's code to notebooks/, then push code + results to GitHub."""
    if notebook:
        import nbformat
        cells = [nbformat.v4.new_code_cell(c) for c in In[1:] if c.strip()]
        nbformat.write(nbformat.v4.new_notebook(cells=cells), f"notebooks/{notebook}")
    run(f'git add results notebooks dpo.py && (git diff --cached --quiet || '
        f'git commit -q -m "{msg}") && git push -q {REPO_URL} HEAD:main')
    print(" Pushed to GitHub")

if not os.path.isdir(REPO_DIR):
    run(f"git clone -q {REPO_URL} {REPO_DIR}")
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
run(f"git remote set-url origin {PLAIN_URL}")
run(f"git pull -q {REPO_URL} main")                       # brings sft.py + Task 3 results
run('git config user.name "kaggle" && git config user.email "kaggle@users.noreply.github.com"')
run("pip uninstall -y -q torchao || true")
run('pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" accelerate')

from huggingface_hub import login
login(token=HF_TOKEN)                                      # needed for the private SFT repo

In [ ]:
get_ipython().run_cell_magic('writefile', 'dpo.py', 'import torch, torch.nn.functional as F\nfrom sft import build_sft_batch, sequence_logprobs\n\ndef build_dpo_batch(tokenizer, prompts, chosen_responses, rejected_responses, max_length):\n\n    return {"chosen": build_sft_batch(tokenizer, prompts, chosen_responses, max_length),\n            "rejected": build_sft_batch(tokenizer, prompts, rejected_responses, max_length)}\n\ndef get_logps(policy, reference, batch):\n\n    pc = sequence_logprobs(policy, batch["chosen"])[0]\n    pr = sequence_logprobs(policy, batch["rejected"])[0]\n    with torch.no_grad():\n        rc = sequence_logprobs(reference, batch["chosen"])[0].to(pc.device)\n        rr = sequence_logprobs(reference, batch["rejected"])[0].to(pc.device)\n    return pc, pr, rc, rr\n\ndef dpo_loss(pc, pr, rc, rr, beta):\n    """loss = -log sigmoid(r_chosen - r_rejected), with rewards r = beta * (policy - reference)."""\n    r_chosen, r_rejected = beta * (pc - rc), beta * (pr - rr)\n    loss = -F.logsigmoid(r_chosen - r_rejected).mean()\n    return loss, r_chosen.detach(), r_rejected.detach()\n\ndef compute_dpo_loss(policy, reference, tokenizer, prompts, chosen, rejected, beta, max_length):\n    batch = build_dpo_batch(tokenizer, prompts, chosen, rejected, max_length)\n    return dpo_loss(*get_logps(policy, reference, batch), beta)[0]\n')

In [ ]:
push("Task 4: DPO code")

In [ ]:
import random, torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import HfApi
import matplotlib.pyplot as plt
from sft import format_prompt
from dpo import build_dpo_batch, get_logps, dpo_loss

DEBUG = False                   # True = tiny test run

N_TRAIN, N_VAL = (64, 16) if DEBUG else (15000, 1000)
MAX_LEN, MAX_PROMPT = 512, 256
ACCUM = 4 if DEBUG else 32          # batch 1 x 32 = effective batch 32
LR, BETA, CLIP = 1e-6, 0.1, 1.0     # fixed lr, no decay
EVAL_EVERY = 4 if DEBUG else 50

SFT_REPO = json.load(open("results/task3_sft/model_location.json"))["hf_repo"]
tok = AutoTokenizer.from_pretrained(SFT_REPO, subfolder="best")
HF_USER = HfApi(token=HF_TOKEN).whoami()["name"]
print("SFT model:", SFT_REPO, "| Hugging Face user:", HF_USER)

In [ ]:
def hh_pairs():
    for ex in load_dataset("Anthropic/hh-rlhf", split="train").shuffle(seed=42):
        c, r = ex["chosen"], ex["rejected"]
        if c.count("\n\nHuman:") == 1 and c.count("\n\nAssistant:") == 1 and r.count("\n\nAssistant:") == 1:
            (q1, a_c), (q2, a_r) = c.split("\n\nAssistant:"), r.split("\n\nAssistant:")
            if q1 == q2:
                yield q1.replace("\n\nHuman:", ""), a_c, a_r

def uf_pairs():
    for ex in load_dataset("HuggingFaceH4/ultrafeedback_binarized", split="train_prefs").shuffle(seed=42):
        if ex["score_chosen"] > ex["score_rejected"]:
            yield ex["prompt"], ex["chosen"][-1]["content"], ex["rejected"][-1]["content"]

def load_pairs(dataset):

    pairs = []
    for q, c, r in {"hh": hh_pairs, "uf": uf_pairs}[dataset]():
        q, c, r = q.strip(), c.strip(), r.strip()
        prompt = format_prompt(q)
        if q and c and r and c != r and len(tok.encode(prompt)) <= MAX_PROMPT:
            pairs.append((prompt, c, r))
        if len(pairs) == N_TRAIN + N_VAL:
            break
    print(f"{dataset}: {len(pairs) - N_VAL} train / {N_VAL} val pairs")
    return pairs[N_VAL:], pairs[:N_VAL]

In [ ]:
def train_dpo(dataset):
    out_dir, log_dir = f"/kaggle/working/outputs/dpo_{dataset}", f"results/task4_dpo_{dataset}"
    os.makedirs(log_dir, exist_ok=True)
    train, val = load_pairs(dataset)

    # Save the exact pairs used (uploaded to Hugging Face with the model)
    os.makedirs(f"{out_dir}/data", exist_ok=True)
    for name, rows in [("train", train), ("val", val)]:
        json.dump([{"prompt": p, "chosen": c, "rejected": r} for p, c, r in rows],
                  open(f"{out_dir}/data/{name}.json", "w"))

    policy = AutoModelForCausalLM.from_pretrained(SFT_REPO, subfolder="best", dtype=torch.float32).to("cuda:0")
    ref = AutoModelForCausalLM.from_pretrained(SFT_REPO, subfolder="best", dtype=torch.float32).to("cuda:1").eval()
    ref.requires_grad_(False)
    opt = torch.optim.AdamW(policy.parameters(), lr=LR, weight_decay=0.0)
    scaler = torch.amp.GradScaler("cuda")
    total = len(train) // ACCUM

    def batch_of(rows):
        return build_dpo_batch(tok, [x[0] for x in rows], [x[1] for x in rows], [x[2] for x in rows], MAX_LEN)

    @torch.no_grad()
    def evaluate():
        policy.eval()
        losses, accs, margins = [], [], []
        for i in range(0, len(val), 4):
            with torch.autocast("cuda", dtype=torch.float16):
                loss, rc, rr = dpo_loss(*get_logps(policy, ref, batch_of(val[i:i + 4])), BETA)
            losses.append(loss.item() * len(rc)); accs += (rc > rr).float().tolist(); margins += (rc - rr).tolist()
        policy.train()
        n = len(accs)
        return {"val_loss": sum(losses) / n, "val_reward_acc": sum(accs) / n, "val_reward_margin": sum(margins) / n}

    def log(rec):
        print(dataset, rec)
        with open(f"{log_dir}/metrics.jsonl", "a") as f:
            f.write(json.dumps(rec) + "\n")

    def save(name):
        policy.save_pretrained(f"{out_dir}/{name}"); tok.save_pretrained(f"{out_dir}/{name}")

    # ---- training loop (1 epoch) ----
    open(f"{log_dir}/metrics.jsonl", "w").close()
    log({"step": 0, **evaluate()})                     # expect acc 0.0, loss 0.693
    random.seed(42); random.shuffle(train)
    best_acc, step, running = -1.0, 0, []
    policy.train()
    for i in range(total * ACCUM):
        with torch.autocast("cuda", dtype=torch.float16):
            loss, rc, rr = dpo_loss(*get_logps(policy, ref, batch_of([train[i]])), BETA)
        scaler.scale(loss / ACCUM).backward()          # gradient accumulation
        running.append((loss.item(), float(rc > rr), (rc - rr).item()))
        if (i + 1) % ACCUM:
            continue
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(policy.parameters(), CLIP)
        scaler.step(opt); scaler.update(); opt.zero_grad()
        step += 1
        if step % 10 == 0 or DEBUG:
            l, a, m = (sum(col) / len(running) for col in zip(*running))
            log({"step": step, "train_loss": l, "train_reward_acc": a, "train_reward_margin": m})
            running = []
        if step % EVAL_EVERY == 0 or step == total:
            metrics = evaluate()
            log({"step": step, **metrics})
            if metrics["val_reward_acc"] > best_acc:
                best_acc = metrics["val_reward_acc"]; save("best")
            if not DEBUG:
                try: push(f"Task 4 DPO {dataset}: step {step}")
                except Exception as e: print("push failed, continuing:", e)
    save("final")

    del policy, ref, opt, scaler                       # free GPU memory before the next run
    gc.collect(); torch.cuda.empty_cache()
    return {"n_train": len(train), "n_val": len(val), "best_val_reward_acc": best_acc}

In [ ]:
def report(dataset, info):
    out_dir, log_dir = f"/kaggle/working/outputs/dpo_{dataset}", f"results/task4_dpo_{dataset}"

    # 1. Curves: DPO loss, reward accuracy, reward margin
    recs = [json.loads(l) for l in open(f"{log_dir}/metrics.jsonl")]
    tr, va = [r for r in recs if "train_loss" in r], [r for r in recs if "val_loss" in r]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, key, title in zip(axes, ["loss", "reward_acc", "reward_margin"], ["DPO loss", "Reward accuracy", "Reward margin"]):
        ax.plot([r["step"] for r in tr], [r[f"train_{key}"] for r in tr], alpha=0.6, label="train (avg 10 steps)")
        ax.plot([r["step"] for r in va], [r[f"val_{key}"] for r in va], "o-", label="validation")
        ax.set(title=f"{title} ({dataset})", xlabel="optimizer step"); ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(f"{log_dir}/dpo_{dataset}_curves.png", dpi=150); plt.show()

    # 2. Sanity check on 3 prompts from this run's evaluation set
    eval_file = {"hh": "eval_sets/hh_rlhf_eval_300_seed42.json", "uf": "eval_sets/alpaca_eval_300_seed42.json"}[dataset]
    model = AutoModelForCausalLM.from_pretrained(f"{out_dir}/best", dtype=torch.float16).to("cuda:0").eval()
    for q in json.load(open(eval_file))[:3]:
        enc = tok(format_prompt(q["instruction"]), return_tensors="pt").to("cuda:0")
        out = model.generate(**enc, do_sample=False, max_new_tokens=256, repetition_penalty=1.1,
                             pad_token_id=tok.pad_token_id)
        print("=" * 80, "\nQ:", q["instruction"], "\nA:", tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)[:400])
    del model; gc.collect(); torch.cuda.empty_cache()

    # 3. Config for the report
    json.dump({"dataset": dataset, "reference": f"{SFT_REPO}/best", **info, "max_len": MAX_LEN, "max_prompt": MAX_PROMPT,
               "batch": 1, "accum": ACCUM, "effective_batch": ACCUM, "lr": LR, "lr_schedule": "fixed", "beta": BETA,
               "optimizer": "AdamW (wd 0)", "clip": CLIP, "epochs": 1,
               "precision": "policy fp32 + fp16 autocast + GradScaler; reference fp32, frozen"},
              open(f"{log_dir}/config.json", "w"), indent=2)

    # 4. Upload model checkpoints + training data to Hugging Face, then push results + notebook to GitHub
    if not DEBUG:
        api = HfApi(token=HF_TOKEN)
        hf_repo = f"{HF_USER}/qwen2.5-0.5b-dpo-{dataset}"
        api.create_repo(hf_repo, private=True, exist_ok=True)
        for name in ["best", "final", "data"]:
            api.upload_folder(folder_path=f"{out_dir}/{name}", path_in_repo=name, repo_id=hf_repo)
            print("Uploaded:", hf_repo, name)
        json.dump({"hf_repo": hf_repo, "best": "best", "final": "final", "data": "data"},
                  open(f"{log_dir}/model_location.json", "w"), indent=2)      # also marks this run as done
        push(f"Task 4: DPO {dataset} finished", notebook="25280065_04_dpo.ipynb")

def already_done(dataset):
    """True if this run already finished earlier (its model_location.json was pushed to GitHub)."""
    return not DEBUG and os.path.exists(f"results/task4_dpo_{dataset}/model_location.json")

In [ ]:
if already_done("hh"):
    print(" HH-RLHF run already finished (found on GitHub), skipping")
else:
    hh_info = train_dpo("hh")
    report("hh", hh_info)
    print(" HH-RLHF done:", hh_info)

In [ ]:
if already_done("uf"):
    print(" UltraFeedback run already finished (found on GitHub), skipping")
else:
    uf_info = train_dpo("uf")
    report("uf", uf_info)
    print(" UltraFeedback done:", uf_info)